In [ ]:
# 1. RÉINSTALLATION (Obligatoire après un redémarrage)
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes

In [ ]:
# ==========================================
# 🧪 TEST DU MODÈLE FINE-TUNÉ (Depuis le Drive)
# ==========================================
from unsloth import FastLanguageModel
from google.colab import drive
import torch

# 1. Montage du Drive (Pour accéder à ton dossier)
drive.mount('/content/drive')



🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Mounted at /content/drive


In [ ]:
 #⚠️ Vérifie bien que ce chemin est celui où tu as sauvegardé ton modèle !
model_path = "/content/drive/MyDrive/PIP_2025-2026_Groupe-1_Concours/Hosni Youssef/Modele_CNRS_FineTuned_QWEN"

print(f"📂 Chargement du modèle depuis : {model_path} ...")

# 2. Chargement du Modèle + Tokenizer
# On n'entraîne plus, on charge juste !
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_path, # <--- C'est ici que la magie opère
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)

# 3. Activation du mode "Inférence" (Lecture seule, plus rapide)
FastLanguageModel.for_inference(model)

print("✅ Modèle chargé avec succès ! Prêt à répondre.")

# ------------------------------------------------------------------
# 4. Fonction de Chat (Pour tester)
# ------------------------------------------------------------------
def ask_jury(question):
    print(f"\n❓ Question : {question}")

    # On prépare le prompt avec le style système
    messages = [
        {"role": "system", "content": "Tu es un membre de jury de concours CNRS strict et professionnel. Réponds aux candidats."},
        {"role": "user", "content": question}
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize = True,
        add_generation_prompt = True,
        return_tensors = "pt",
    ).to("cuda")

    # Génération
    outputs = model.generate(
        input_ids = inputs,
        max_new_tokens = 300,
        use_cache = True,
        temperature = 0.1, # Très bas pour être précis
        do_sample = True
    )

    # Décodage
    response = tokenizer.batch_decode(outputs)[0]

    # Nettoyage du texte (on enlève le prompt)
    if "assistant\n" in response:
         clean_response = response.split("assistant\n")[-1].replace("<|im_end|>", "").strip()
    else:
         clean_response = response

    print(f"🤖 Réponse :\n{clean_response}")

# ------------------------------------------------------------------
# 5. Lancement des Tests
# ------------------------------------------------------------------
ask_jury("Quelles sont les conditions de diplôme pour le concours Ingénieur de Recherche ?")
ask_jury("Bonjour, comment ça va ?") # Pour voir s'il reste pro ou s'il bavarde

📂 Chargement du modèle depuis : /content/drive/MyDrive/PIP_2025-2026_Groupe-1_Concours/Hosni Youssef/Modele_CNRS_FineTuned_QWEN ...
==((====))==  Unsloth 2026.1.2: Fast Qwen2 patching. Transformers: 4.57.3.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.55G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

Unsloth 2026.1.2 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


✅ Modèle chargé avec succès ! Prêt à répondre.

❓ Question : Quelles sont les conditions de diplôme pour le concours Ingénieur de Recherche ?
🤖 Réponse :
Pour l'ingénieur de recherche (IR), le jury attend une **connaissance approfondie** des grilles de notation. Le **niveau minimal** est un doctorat (ou équivalent). Des **conseils méthodologiques** sans justification technique ne sont pas pris en compte.

❓ Question : Bonjour, comment ça va ?
🤖 Réponse :
Je vais bien en tant que jury. Comment l'candidates se prépare-t-il pour son audition ?


In [ ]:
# ==========================================
# 💬 INTERFACE DE CHAT INTERACTIVE
# ==========================================

# On définit le rôle du robot (Jury CNRS)
SYSTEM_PROMPT = "Tu es un membre de jury de concours CNRS strict, précis et professionnel. Tu réponds aux candidats en te basant sur les règles administratives."

print("🤖 LE JURY EST PRÊT ! (Tape 'exit' pour arrêter)")
print("-" * 50)

while True:
    # 1. On demande ta question
    user_input = input("\n🗣️ VOTRE QUESTION : ")

    # Pour arrêter la discussion
    if user_input.lower() in ["exit", "quit", "stop"]:
        print("👋 Fin de la session.")
        break

    # 2. Préparation du message
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_input}
    ]

    # 3. Encodage et envoi au GPU
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize = True,
        add_generation_prompt = True,
        return_tensors = "pt",
    ).to("cuda")

    # 4. Génération de la réponse
    outputs = model.generate(
        input_ids = inputs,
        max_new_tokens = 300,   # Longueur max de la réponse
        use_cache = True,
        temperature = 0.1,      # 0.1 = Très sérieux / 0.8 = Créatif
        do_sample = True
    )

    # 5. Nettoyage et affichage
    response_text = tokenizer.batch_decode(outputs)[0]

    # On coupe pour ne garder que la réponse de l'assistant
    if "assistant\n" in response_text:
        final_response = response_text.split("assistant\n")[-1].replace("<|im_end|>", "").strip()
    else:
        # Fallback pour Llama 3 si le format diffère légèrement
        final_response = response_text.split(user_input)[-1].replace("<|eot_id|>", "").strip()

    print(f"\n⚖️ JURY : {final_response}")
    print("-" * 50)

🤖 LE JURY EST PRÊT ! (Tape 'exit' pour arrêter)
--------------------------------------------------

🗣️ VOTRE QUESTION : bonjour

⚖️ JURY : Oui, quelle est l'activité à laquelle vous souhaitez vous consacrer ? Conformément aux règles du CNRS, je dois vous orienter vers votre poste de travail.
--------------------------------------------------

🗣️ VOTRE QUESTION : je suis un ingénieur biologiste quel concours me convient ?

⚖️ JURY : En tant que jury, je vous invite à préciser votre profil (niveau de poste, localisation...). Le CNRS adapte son recrutement aux besoins des unités selon deux profils : les **fonctionnaires** (CDI, 70% de la place) et les **contrat de droit privé** (30%).
--------------------------------------------------

🗣️ VOTRE QUESTION : Combien de postes sont disponibles pour le concours ingénieur biologiste en analyse des données ?

⚖️ JURY : Conformément aux dernières dispositions du poste 120, il n'y a actuellement aucune offre pour ce profil de poste spécifique.
---

In [ ]:
# ==========================================
# 📂 GÉNÉRATION SUR FICHIER DRIVE SPÉCIFIQUE
# ==========================================
from google.colab import drive
import pandas as pd
from tqdm import tqdm
import os

# 1. Montage du Drive
drive.mount('/content/drive')

# --- CONFIGURATION ---
# Ton fichier d'entrée exact
INPUT_PATH = "/content/drive/MyDrive/PIP_2025-2026_Groupe-1_Concours/Mohamed-Taha Belhaj - Analyse/jeu_test.csv"

# Le dossier où se trouve le fichier (pour y mettre la sortie)
OUTPUT_DIR = os.path.dirname(INPUT_PATH)
# Le nom du fichier de sortie demandé
OUTPUT_PATH = os.path.join(OUTPUT_DIR, "reponses_assistant.csv")

# Le nom de ta colonne
COL_QUESTION = "Question"

# -------------------------------------

print(f"📂 Lecture du fichier : {INPUT_PATH}...")

if not os.path.exists(INPUT_PATH):
    print("❌ ERREUR : Le chemin est introuvable. Vérifie que le Drive est bien monté et le chemin exact.")
else:
    # 2. Chargement
    # sep=None et engine='python' permettent de détecter automatiquement ; ou ,
    df = pd.read_csv(INPUT_PATH, sep=None, engine='python', encoding='utf-8-sig')
    print(f"✅ Fichier chargé ! {len(df)} lignes trouvées.")

    if COL_QUESTION not in df.columns:
        print(f"❌ ERREUR : La colonne '{COL_QUESTION}' est absente.")
        print(f"👀 Colonnes disponibles : {list(df.columns)}")
    else:
        print("\n🚀 Démarrage de l'analyse du Jury...")
        tqdm.pandas()

        # Fonction sécurisée pour appeler ton modèle
        def safe_answer(q):
            if pd.isna(q) or str(q).strip() == "":
                return ""
            try:
                # Appel de ta fonction answer()
                res = answer(str(q))
                # Si answer() renvoie un dictionnaire (mode RAG), on prend juste le texte
                if isinstance(res, dict):
                    return res['response']
                return res
            except Exception as e:
                return f"Erreur : {e}"

        # 3. Génération des réponses
        df['Reponse_Assistant'] = df[COL_QUESTION].progress_apply(safe_answer)

        # 4. Sauvegarde
        print(f"\n💾 Sauvegarde en cours vers : {OUTPUT_PATH}")
        df.to_csv(OUTPUT_PATH, index=False, sep=';', encoding='utf-8-sig')

        print("🎉 TERMINÉ ! Le fichier 'reponses_assistant.csv' est enregistré dans le même dossier que ton jeu de test.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📂 Lecture du fichier : /content/drive/MyDrive/PIP_2025-2026_Groupe-1_Concours/Mohamed-Taha Belhaj - Analyse/jeu_test.csv...
✅ Fichier chargé ! 48 lignes trouvées.

🚀 Démarrage de l'analyse du Jury...


100%|██████████| 48/48 [00:00<00:00, 16270.13it/s]


💾 Sauvegarde en cours vers : /content/drive/MyDrive/PIP_2025-2026_Groupe-1_Concours/Mohamed-Taha Belhaj - Analyse/reponses_assistant.csv
🎉 TERMINÉ ! Le fichier 'reponses_assistant.csv' est enregistré dans le même dossier que ton jeu de test.
